# MetalBind-only reproduction — TestSet300 + TestSetNew46

This notebook runs **only MetalBind**. It uses the same persistent folders as the master notebook, so previously completed ESM-C, PDB normalization, per-protein preprocessing, and per-ion predictions are reused whenever they pass validation.

Final output: a compact MetalBind result table for TestSet300 and TestSetNew46 plus a hard 13-ion × 2-dataset completion guard.


In [12]:
# ============================================================
# 0. COLAB / DRIVE / PATHS / SWITCHES
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, shutil, subprocess, time, hashlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "scipy", "scikit-learn", "biopython", "tqdm"],
    check=True
)

import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
WORK = BASE/'baseline_reproduction_master_v6'
REPOS = WORK/'repos'
CACHE = WORK/'cache'
PRED = WORK/'predictions'
RESULTS = WORK/'results'
MANIFEST = WORK/'manifest'
CANON = WORK/'canonical_inputs'

for p in [WORK, REPOS, CACHE, PRED, RESULTS, MANIFEST, CANON]:
    p.mkdir(parents=True, exist_ok=True)

TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

RUN_METALBIND_TRANSFER = True
RUN_METALBIND_NATIVE = False
FAST_RESUME = True

REBUILD_METALBIND_ESMC = False
REBUILD_METALBIND_SURFACE = False
REBUILD_METALBIND_INFERENCE = False

METALBIND_KEOPS_MODE = 'auto'

FIXED_THRESHOLD = 0.50
BOOTSTRAP_REPS = 2000
RANDOM_SEED = 42

METALB_IONS = ['CA','CD','CO','CU','CU1','FE','FE2','K','MG','MN','NA','NI','ZN']

METALB_EXT = WORK/'metalbind_external_v6'
METALB_EXT.mkdir(parents=True, exist_ok=True)

METALBIND_EXTERNAL_ROOT = WORK/'metalbind_external_predictions_v6'
METALBIND_EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)

print('BASE    =', BASE)
print('WORK    =', WORK)
print('RESULTS =', RESULTS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE    = /content/drive/MyDrive/Protein-binding-Phi
WORK    = /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6
RESULTS = /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/results


In [13]:
# ============================================================
# 1. LOAD ONLY THE TWO EVALUATION SETS + METAL LABELS
# ============================================================
def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_frame(path):
    if not Path(path).exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)

    required = {'prot_id', 'sequence', 'metal_binding'}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f'{path}: missing columns {sorted(missing)}')

    df['metal_binding'] = df['metal_binding'].apply(parse_list)

    if not df.prot_id.is_unique:
        raise RuntimeError(f'{path}: duplicate prot_id values')

    for r in df.itertuples():
        if len(r.sequence) != len(r.metal_binding):
            raise RuntimeError(
                f'{path}/{r.prot_id}: sequence length {len(r.sequence)} '
                f'!= metal label length {len(r.metal_binding)}'
            )

    return df

test300_df = load_frame(TEST300_CSV)
test46_df = load_frame(TEST46_CSV)

DATASETS = {
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}

def write_fasta(df, path):
    with open(path, 'w') as f:
        for r in df.itertuples():
            f.write(f'>{r.prot_id}\n{r.sequence}\n')

for dataset, df in DATASETS.items():
    write_fasta(df, CANON/f'{dataset}.fasta')

audit_rows = []
for dataset, df in DATASETS.items():
    nres = int(sum(len(x) for x in df.metal_binding))
    pos = int(sum(sum(x) for x in df.metal_binding))
    audit_rows.append([dataset, len(df), nres, pos, nres-pos, pos/max(nres,1)])

dataset_audit = pd.DataFrame(
    audit_rows,
    columns=['dataset','proteins','residues','metal_positive','metal_negative','prevalence']
)
display(dataset_audit)


,dataset,proteins,residues,metal_positive,metal_negative,prevalence
0,TestSet300,300,62689,881,61808,0.014054
1,TestSetNew46,46,6227,77,6150,0.012366


In [14]:
# ============================================================
# 2. PERSISTENT METALBIND CACHE PREFLIGHT
# ============================================================
def _simple_prob_csv_complete(path, df, require_surface_column=True):
    path = Path(path)
    if not path.exists():
        return False

    try:
        z = pd.read_csv(path)
    except Exception:
        return False

    need = {'prot_id','residue_index','probability'}
    if require_surface_column:
        need.add('is_surface_residue')

    if not need.issubset(z.columns):
        return False

    z['prot_id'] = z['prot_id'].astype(str)

    expected_rows = int(sum(len(r.sequence) for r in df.itertuples()))
    if len(z) != expected_rows:
        return False

    if set(z.prot_id) != set(df.prot_id.astype(str)):
        return False

    for r in df.itertuples():
        q = z[z.prot_id.eq(str(r.prot_id))].sort_values('residue_index')

        if len(q) != len(r.sequence):
            return False

        idx = q.residue_index.to_numpy(int)
        if not np.array_equal(idx, np.arange(len(r.sequence))):
            return False

        p = q.probability.to_numpy(float)
        if not np.all(np.isfinite(p)) or np.min(p) < 0 or np.max(p) > 1:
            return False

    return True

cache_rows = []

for dataset, df in DATASETS.items():
    root = METALBIND_EXTERNAL_ROOT/dataset

    for ion in METALB_IONS:
        fp = root/f'{ion}.csv.gz'
        ok = (
            _simple_prob_csv_complete(fp, df, require_surface_column=True)
            and not REBUILD_METALBIND_INFERENCE
        )
        cache_rows.append([dataset, ion, ok, str(fp)])

metalbind_cache_audit = pd.DataFrame(
    cache_rows,
    columns=['dataset','ion','valid_cached_prediction','path']
)

display(
    metalbind_cache_audit
    .groupby('dataset')['valid_cached_prediction']
    .agg(['sum','count'])
    .reset_index()
)

METALBIND_FINAL_CACHE_COMPLETE = bool(
    metalbind_cache_audit.valid_cached_prediction.all()
)

if METALBIND_FINAL_CACHE_COMPLETE:
    print(
        'MetalBind final prediction cache COMPLETE: '
        '26/26 dataset-ion outputs valid -> setup/preprocessing/inference SKIP.'
    )
else:
    print(
        'MetalBind final prediction cache INCOMPLETE: '
        f'{int(metalbind_cache_audit.valid_cached_prediction.sum())}/26 valid ion files. '
        'Only missing/invalid work will be generated.'
    )


,dataset,sum,count
0,TestSet300,13,13
1,TestSetNew46,13,13


MetalBind final prediction cache COMPLETE: 26/26 dataset-ion outputs valid -> setup/preprocessing/inference SKIP.


## 3. Exact evaluator from the master reproduction notebook


In [15]:
# ============================================================
# 5A. FULL METRIC IMPLEMENTATION
# ============================================================
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    f1_score, fbeta_score, matthews_corrcoef, precision_score, recall_score,
    accuracy_score, balanced_accuracy_score, jaccard_score, brier_score_loss,
    auc, roc_curve
)
from scipy.stats import t as student_t

EPS=1e-15

def confusion(y,pred):
    y=np.asarray(y,np.int8); pred=np.asarray(pred,np.int8)
    tp=int(np.sum((y==1)&(pred==1))); fp=int(np.sum((y==0)&(pred==1)))
    tn=int(np.sum((y==0)&(pred==0))); fn=int(np.sum((y==1)&(pred==0)))
    return tp,fp,tn,fn

def safe_div(a,b): return float(a/b) if b else 0.0

def ci_legacy(x):
    x=np.asarray(x,float)
    return np.nan if len(x)<2 else float(np.std(x,ddof=0)/np.sqrt(len(x))*student_t.ppf(.975,len(x)))

def ci_standard(x):
    x=np.asarray(x,float)
    return np.nan if len(x)<2 else float(np.std(x,ddof=1)/np.sqrt(len(x))*student_t.ppf(.975,len(x)-1))

def bootstrap_mean_ci(x,reps=2000,seed=42):
    x=np.asarray(x,float)
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed)
    vals=np.empty(reps,float)
    for i in range(reps): vals[i]=np.mean(rng.choice(x,size=len(x),replace=True))
    return tuple(np.percentile(vals,[2.5,97.5]).astype(float))

def threshold_free_metrics(y,p):
    y=np.asarray(y,np.int8); p=np.asarray(p,float)
    pr,rc,_=precision_recall_curve(y,p)
    # recall is returned descending; reverse before trapezoidal integration.
    pr_auc=float(auc(rc[::-1],pr[::-1]))
    return {
        'n_residues':int(len(y)),
        'n_positive':int(y.sum()),
        'n_negative':int(len(y)-y.sum()),
        'prevalence':float(y.mean()),
        'AP_AUPRC':float(average_precision_score(y,p)),
        'PR_AUC_trapezoid':pr_auc,
        'AUROC':float(roc_auc_score(y,p)),
        'Brier':float(brier_score_loss(y,p)),
    }

def global_threshold_metrics(y,p,t):
    y=np.asarray(y,np.int8); p=np.asarray(p,float); pred=(p>=t).astype(np.int8)
    tp,fp,tn,fn=confusion(y,pred)
    spec=safe_div(tn,tn+fp); npv=safe_div(tn,tn+fn)
    return {
        'threshold':float(t),'TP':tp,'FP':fp,'TN':tn,'FN':fn,
        'global_accuracy':float(accuracy_score(y,pred)),
        'global_balanced_accuracy':float(balanced_accuracy_score(y,pred)),
        'global_precision_PPV':float(precision_score(y,pred,zero_division=0)),
        'global_recall_sensitivity':float(recall_score(y,pred,zero_division=0)),
        'global_specificity':spec,'global_NPV':npv,
        'global_positive_F1':float(f1_score(y,pred,average='binary',pos_label=1,zero_division=0)),
        'global_F1':float(f1_score(y,pred,average='binary',pos_label=1,zero_division=0)),  # legacy alias
        'negative_class_F1':float(f1_score(y,pred,average='binary',pos_label=0,zero_division=0)),
        'class_macro_F1':float(f1_score(y,pred,average='macro',zero_division=0)),
        'class_weighted_F1':float(f1_score(y,pred,average='weighted',zero_division=0)),
        'class_micro_F1':float(f1_score(y,pred,average='micro',zero_division=0)),
        'global_F2':float(fbeta_score(y,pred,beta=2,zero_division=0)),
        'global_Jaccard':float(jaccard_score(y,pred,zero_division=0)),
        'global_MCC':float(matthews_corrcoef(y,pred)) if len(np.unique(pred))>1 else 0.0,
        'predicted_positive_rate':float(pred.mean()),
    }

def per_protein_table(df,probs,label,t):
    rows=[]
    for r in df.itertuples():
        pid=str(r.prot_id); y=np.asarray(getattr(r,label),np.int8); p=np.asarray(probs[pid],float)
        assert len(y)==len(p),(pid,len(y),len(p))
        pred=(p>=t).astype(np.int8); tp,fp,tn,fn=confusion(y,pred)
        include=(tp+fp+fn)>0
        false_alarm_only=((tp+fn)==0 and fp>0)
        precision=safe_div(tp,tp+fp); recall=safe_div(tp,tp+fn); specificity=safe_div(tn,tn+fp)
        f1=safe_div(2*tp,2*tp+fp+fn)
        den=(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
        mcc=float((tp*tn-fp*fn)/np.sqrt(den)) if den>0 else 0.0
        accuracy=safe_div(tp+tn,tp+fp+tn+fn)
        rows.append([pid,include,false_alarm_only,tp,fp,tn,fn,precision,recall,specificity,f1,mcc,accuracy])
    return pd.DataFrame(rows,columns=[
        'prot_id','included_macro_positive','false_alarm_only','TP','FP','TN','FN',
        'precision','recall','specificity','F1','MCC','accuracy'
    ])

def per_protein_summary(pt,reps=2000,seed=42):
    inc=pt[pt.included_macro_positive].copy()
    out={
        'protein_macro_positive_n':int(len(inc)),
        'protein_TN_only_excluded_n':int((~pt.included_macro_positive).sum()),
        'protein_false_alarm_only_n':int(pt.false_alarm_only.sum()),
    }
    for metric in ['precision','recall','specificity','F1','MCC','accuracy']:
        x=inc[metric].to_numpy(float)
        out[f'protein_mean_{metric}']=float(np.mean(x)) if len(x) else np.nan
        out[f'protein_{metric}_CI95_legacy_halfwidth']=ci_legacy(x)
        out[f'protein_{metric}_CI95_standard_halfwidth']=ci_standard(x)
        if metric in ['F1','MCC']:
            lo,hi=bootstrap_mean_ci(x,reps,seed)
            out[f'protein_{metric}_bootstrap95_low']=lo
            out[f'protein_{metric}_bootstrap95_high']=hi
    return out

def evaluate_probability_dict(df,probs,label,method,dataset,operating_point,t,comparison_tier):
    ids=[str(x) for x in df.prot_id]
    missing=[pid for pid in ids if pid not in probs]
    assert not missing,f'{method}/{dataset}/{label}: missing {missing[:10]}'
    y=np.concatenate([np.asarray(v,np.int8) for v in df[label]])
    p=np.concatenate([np.asarray(probs[str(pid)],float) for pid in df.prot_id])
    tf=threshold_free_metrics(y,p)
    gt=global_threshold_metrics(y,p,t)
    pt=per_protein_table(df,probs,label,t)
    ps=per_protein_summary(pt,BOOTSTRAP_REPS,RANDOM_SEED)
    out={'method':method,'dataset':dataset,'label':label,'comparison_tier':comparison_tier,
         'operating_point':operating_point,**tf,**gt,**ps}
    return out,pt

def save_probs(probs,path):
    rows=[]
    for pid,p in probs.items():
        for i,v in enumerate(np.asarray(p,float)): rows.append((pid,i,float(v)))
    pd.DataFrame(rows,columns=['prot_id','residue_index','probability']).to_csv(path,index=False,compression='gzip')


In [16]:
# ============================================================
# 4. LAZY MICROMAMBA HELPERS + PIN ONLY METALBIND
# ============================================================
MAMBA = Path('/usr/local/bin/micromamba')
LOCAL_MAMBA_ROOT = Path('/content/micromamba_binding_baselines')
LOCAL_MAMBA_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['MAMBA_ROOT_PREFIX'] = str(LOCAL_MAMBA_ROOT)

def sh(cmd, cwd=None, check=True, capture=False):
    print('+', cmd)
    return subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )

def ensure_micromamba():
    if MAMBA.exists():
        return

    sh(
        'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
        '| tar -xvj -C /tmp bin/micromamba'
    )
    sh('cp /tmp/bin/micromamba /usr/local/bin/micromamba')
    sh('chmod +x /usr/local/bin/micromamba')

def env_path(name):
    return LOCAL_MAMBA_ROOT/'envs'/name

def env_healthy(name):
    ensure_micromamba()
    p = env_path(name)

    if not p.exists():
        return False

    r = subprocess.run(
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
        f'micromamba run -n {name} python -c "import sys; print(sys.executable)"',
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    return r.returncode == 0

def create_env(name, python='3.10'):
    ensure_micromamba()

    if env_healthy(name):
        print('REUSE LOCAL ENV', name)
        return

    p = env_path(name)
    if p.exists():
        print('REMOVE BROKEN LOCAL ENV', name)
        sh(f'rm -rf "{p}"')

    sh(
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
        f'micromamba create -y -n {name} -c conda-forge python={python} pip'
    )

    if not env_healthy(name):
        raise RuntimeError(f'Environment {name} was created but cannot execute Python.')

def mex(name, cmd, cwd=None, check=True, capture=False):
    ensure_micromamba()
    prefix = (
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
        f'micromamba run -n {name}'
    )
    return sh(
        f'{prefix} {cmd}',
        cwd=cwd,
        check=check,
        capture=capture,
    )

METALB = REPOS/'metalbind'
METALB_URL = 'https://github.com/zpliulab/MetalBind.git'
METALB_SHA = '6d2d13c4566477a1b79ac8e22ac1aaa6840c25a0'

if not METALBIND_FINAL_CACHE_COMPLETE:
    if not METALB.exists():
        sh(f'git clone {METALB_URL} "{METALB}"')

    if not (METALB/'.git').exists():
        raise RuntimeError(f'Existing MetalBind path is not a git repository: {METALB}')

    sh('git fetch --all --tags', cwd=METALB)
    sh(f'git reset --hard {METALB_SHA}', cwd=METALB)

    got = subprocess.check_output(
        'git rev-parse HEAD',
        cwd=METALB,
        shell=True,
        text=True
    ).strip()

    if got != METALB_SHA:
        raise RuntimeError(f'MetalBind pin mismatch: {got} != {METALB_SHA}')

    print('MetalBind pinned:', got)
else:
    print('Repository setup SKIP: final prediction cache already complete.')


Repository setup SKIP: final prediction cache already complete.


In [17]:
# ============================================================
# 5. RESUME-SAFE EXACT-CHAIN-A STRUCTURE NORMALIZATION
# ============================================================
NORMALIZED_EXTERNAL_PDB = {}

if not METALBIND_FINAL_CACHE_COMPLETE:
    from copy import deepcopy
    from Bio.PDB import PDBParser, PDBIO
    from Bio.PDB.Structure import Structure
    from Bio.PDB.Model import Model
    from Bio.PDB.Chain import Chain
    from Bio.SeqUtils import seq1

    AA3 = {
        'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
        'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
    }

    METAL_EXTERNAL_STRUCT_ROOT = CACHE/'metal_external_normalized_structures'
    METAL_EXTERNAL_STRUCT_ROOT.mkdir(parents=True, exist_ok=True)

    PDB_CANDIDATES = [
        BASE/'pdb/testset300',
        BASE/'pdb/testset46',
        BASE/'esmFold_pdb_files',
        BASE/'pdb/all',
    ]

    def _standard_chain_residues(chain):
        return [
            r for r in chain
            if r.id[0] == ' ' and r.resname.strip().upper() in AA3
        ]

    def _residue_seq(residues):
        return ''.join(seq1(r.resname.strip().upper()) for r in residues)

    def _cached_normalized_ok(path, target_sequence):
        path = Path(path)

        if not path.exists() or path.stat().st_size == 0:
            return False

        try:
            st = PDBParser(QUIET=True).get_structure('cached', str(path))
            model = next(st.get_models())

            if 'A' not in model:
                return False

            seq = _residue_seq(_standard_chain_residues(model['A']))
            return seq == target_sequence

        except Exception:
            return False

    def find_project_pdb(pid):
        pid = str(pid)

        for root in PDB_CANDIDATES:
            if not root.exists():
                continue

            p = root/f'{pid}.pdb'
            if p.exists():
                return p

            hit = list(root.rglob(f'{pid}.pdb'))
            if hit:
                return hit[0]

        return None

    def resolve_external_structure(dataset, pid):
        p = find_project_pdb(pid)

        if p is not None:
            return Path(p)

        # Existing bindNode24 checkout may provide TestSet300 AlphaFold structures.
        # No bindNode24 model or inference is used in this notebook.
        if dataset == 'TestSet300':
            q = REPOS/'bindnode24'/'dataset'/'AlphaFold_predictions'/f'{pid}.pdb'
            if q.exists():
                return q

        return None

    def normalize_pdb_exact_chain_A(src, target_sequence, dst):
        src = Path(src)
        dst = Path(dst)

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('src', str(src))
        model = next(structure.get_models())

        exact = []
        diagnostic = []

        for chain in model:
            residues = _standard_chain_residues(chain)
            seq = _residue_seq(residues)

            diagnostic.append((str(chain.id), len(seq), seq[:20]))

            if seq == target_sequence:
                exact.append((chain, residues))

        if not exact:
            raise RuntimeError(
                f'{src}: no chain exactly matches canonical sequence length '
                f'{len(target_sequence)}. Chains={diagnostic}'
            )

        exact = sorted(
            exact,
            key=lambda x: (0 if str(x[0].id) == 'A' else 1, str(x[0].id))
        )

        chain, residues = exact[0]

        out_structure = Structure('normalized')
        out_model = Model(0)
        out_chain = Chain('A')

        out_structure.add(out_model)
        out_model.add(out_chain)

        for i, res in enumerate(residues, start=1):
            rr = deepcopy(res)
            rr.detach_parent()
            rr.id = (' ', i, ' ')
            out_chain.add(rr)

        dst.parent.mkdir(parents=True, exist_ok=True)

        io = PDBIO()
        io.set_structure(out_structure)
        io.save(str(dst))

        if not _cached_normalized_ok(dst, target_sequence):
            raise RuntimeError(f'{dst}: normalized read-back sequence mismatch')

        return dst, str(chain.id), len(exact)

    structure_audit = []

    for dataset, df in DATASETS.items():
        NORMALIZED_EXTERNAL_PDB[dataset] = {}
        droot = METAL_EXTERNAL_STRUCT_ROOT/dataset
        droot.mkdir(parents=True, exist_ok=True)

        for r in df.itertuples():
            pid = str(r.prot_id)
            dst = droot/f'{pid}.pdb'

            if _cached_normalized_ok(dst, r.sequence):
                NORMALIZED_EXTERNAL_PDB[dataset][pid] = dst

                structure_audit.append([
                    dataset, pid, 'CACHED', 'A', 1,
                    str(dst), len(r.sequence), 'CACHED_PASS'
                ])
                continue

            src = resolve_external_structure(dataset, pid)

            if src is None:
                raise FileNotFoundError(
                    f'{dataset}/{pid}: no source PDB found and normalized cache is absent. '
                    f'Searched project roots: {[str(x) for x in PDB_CANDIDATES]}'
                )

            dst, source_chain, n_exact = normalize_pdb_exact_chain_A(
                src, r.sequence, dst
            )

            NORMALIZED_EXTERNAL_PDB[dataset][pid] = dst

            structure_audit.append([
                dataset, pid, str(src), source_chain, n_exact,
                str(dst), len(r.sequence), 'PASS'
            ])

    structure_audit_df = pd.DataFrame(
        structure_audit,
        columns=[
            'dataset','prot_id','source_pdb','selected_source_chain',
            'n_exact_sequence_matching_chains','normalized_pdb','length','status'
        ]
    )

    structure_audit_df.to_csv(
        RESULTS/'MetalBind_structure_normalization_audit.csv',
        index=False
    )

    display(
        structure_audit_df
        .groupby(['dataset','status'])
        .size()
        .reset_index(name='n')
    )

    for dataset, df in DATASETS.items():
        if len(NORMALIZED_EXTERNAL_PDB[dataset]) != len(df):
            raise RuntimeError(f'{dataset}: incomplete normalized structure map')

    print('MetalBind exact-chain-A structures: PASS.')

else:
    print('Structure normalization SKIP: final prediction cache already complete.')


Structure normalization SKIP: final prediction cache already complete.


## 6. MetalBind author-compatible environment


In [18]:
# ============================================================
# RESUME HELPER AFTER COLAB DISCONNECT
# ============================================================

def _simple_prob_csv_complete(path, df, require_surface_column=False):
    path = Path(path)

    if not path.exists():
        return False

    try:
        z = pd.read_csv(path)
    except Exception:
        return False

    need = {
        'prot_id',
        'residue_index',
        'probability'
    }

    if require_surface_column:
        need.add('is_surface_residue')

    if not need.issubset(z.columns):
        return False

    z['prot_id'] = z['prot_id'].astype(str)

    expected_rows = int(
        sum(len(r.sequence) for r in df.itertuples())
    )

    if len(z) != expected_rows:
        return False

    if set(z.prot_id) != set(df.prot_id.astype(str)):
        return False

    for r in df.itertuples():
        pid = str(r.prot_id)
        L = len(r.sequence)

        q = (
            z[z.prot_id.eq(pid)]
            .sort_values('residue_index')
        )

        if len(q) != L:
            return False

        idx = q.residue_index.to_numpy(int)

        if not np.array_equal(idx, np.arange(L)):
            return False

        p = q.probability.to_numpy(float)

        if (
            not np.all(np.isfinite(p))
            or np.min(p) < 0
            or np.max(p) > 1
        ):
            return False

    return True

In [19]:
# ============================================================
# 18A. MetalBind AUTHOR-COMPATIBLE ENVIRONMENT — RESUME SAFE
# ============================================================
METALB=REPOS/'metalbind'
METALB_IONS=['CA','CD','CO','CU','CU1','FE','FE2','K','MG','MN','NA','NI','ZN']
METALB_EXT=WORK/'metalbind_external_v6'
METALB_EXT.mkdir(parents=True,exist_ok=True)
METALBIND_EXTERNAL_ROOT=WORK/'metalbind_external_predictions_v6'
METALBIND_EXTERNAL_ROOT.mkdir(parents=True,exist_ok=True)


def _metalbind_complete_persistent_cache():
    if not RUN_METALBIND_TRANSFER or not FAST_RESUME or REBUILD_METALBIND_INFERENCE:
        return False
    for dataset,df in DATASETS.items():
        root=METALBIND_EXTERNAL_ROOT/dataset
        for ion in METALB_IONS:
            if not _simple_prob_csv_complete(root/f'{ion}.csv.gz',df,require_surface_column=True):
                return False
    return True


METALBIND_FINAL_CACHE_COMPLETE=_metalbind_complete_persistent_cache()
METALBIND_RUNTIME_NEEDED=(
    RUN_METALBIND_NATIVE
    or (RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE)
)

if METALBIND_FINAL_CACHE_COMPLETE:
    print('MetalBind: all 13 ions x Test300/Test46 strictly cached -> preprocessing + inference can be skipped.')

metalbind_envlib=LOCAL_MAMBA_ROOT/'envs'/'metalbind'/'lib'


def metal_mex(cmd,cwd=None,check=True,capture=False):
    """Run MetalBind with CUDA-11.3 shared libs without contaminating other envs."""
    old_ld=os.environ.get('LD_LIBRARY_PATH')
    old_omp=os.environ.get('OMP_NUM_THREADS')
    old_mkl=os.environ.get('MKL_NUM_THREADS')
    try:
        lib=str(metalbind_envlib)
        os.environ['LD_LIBRARY_PATH']=lib + (':' + old_ld if old_ld else '')
        n=max(1,min(8,os.cpu_count() or 1))
        os.environ['OMP_NUM_THREADS']=str(n)
        os.environ['MKL_NUM_THREADS']=str(n)
        return mex('metalbind',cmd,cwd=cwd,check=check,capture=capture)
    finally:
        if old_ld is None: os.environ.pop('LD_LIBRARY_PATH',None)
        else: os.environ['LD_LIBRARY_PATH']=old_ld
        if old_omp is None: os.environ.pop('OMP_NUM_THREADS',None)
        else: os.environ['OMP_NUM_THREADS']=old_omp
        if old_mkl is None: os.environ.pop('MKL_NUM_THREADS',None)
        else: os.environ['MKL_NUM_THREADS']=old_mkl


def metal_preprocess_mex(cmd,cwd=None,check=True,capture=False):
    """Run surface/KeOps preprocessing without forcing the legacy CUDA-11.3 runtime.

    MetalBind inference needs the original torch/PyG cu113 stack, but its NumPy/KeOps
    surface preprocessing does not. On newer Colab GPUs, forcing CUDA 11.3 NVRTC can
    reject the GPU architecture. This runner lets KeOps see the host CUDA toolchain;
    the exact same formula is used, with CPU fallback selected separately if needed.
    """
    old_ld=os.environ.get('LD_LIBRARY_PATH')
    old_cuda_path=os.environ.get('CUDA_PATH')
    old_omp=os.environ.get('OMP_NUM_THREADS')
    old_mkl=os.environ.get('MKL_NUM_THREADS')
    try:
        if old_ld:
            parts=[x for x in old_ld.split(':') if Path(x).resolve()!=metalbind_envlib.resolve()]
            if parts: os.environ['LD_LIBRARY_PATH']=':'.join(parts)
            else: os.environ.pop('LD_LIBRARY_PATH',None)
        if Path('/usr/local/cuda').exists():
            os.environ['CUDA_PATH']='/usr/local/cuda'
        n=max(1,min(8,os.cpu_count() or 1))
        os.environ['OMP_NUM_THREADS']=str(n)
        os.environ['MKL_NUM_THREADS']=str(n)
        return mex('metalbind',cmd,cwd=cwd,check=check,capture=capture)
    finally:
        if old_ld is None: os.environ.pop('LD_LIBRARY_PATH',None)
        else: os.environ['LD_LIBRARY_PATH']=old_ld
        if old_cuda_path is None: os.environ.pop('CUDA_PATH',None)
        else: os.environ['CUDA_PATH']=old_cuda_path
        if old_omp is None: os.environ.pop('OMP_NUM_THREADS',None)
        else: os.environ['OMP_NUM_THREADS']=old_omp
        if old_mkl is None: os.environ.pop('MKL_NUM_THREADS',None)
        else: os.environ['MKL_NUM_THREADS']=old_mkl


if METALBIND_RUNTIME_NEEDED:
    create_env('metalbind','3.7')

    sh(
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" micromamba install -y -n metalbind '
        '-c conda-forge -c bioconda '
        'scripps-msms=2.6.1 reduce "cudatoolkit=11.3"'
    )

    mex('metalbind',"python -m pip install -q --upgrade 'pip==23.3.2' 'setuptools<69' wheel")
    mex('metalbind',
        "python -m pip install -q 'numpy==1.21.6' 'scipy==1.7.3' 'scikit-learn==1.0.2' "
        "'biopython==1.78' 'pandas==1.3.5' 'tqdm<5'"
    )
    mex('metalbind',
        "python -m pip install -q 'torch==1.10.1+cu113' "
        "-f https://download.pytorch.org/whl/torch_stable.html"
    )
    mex('metalbind',
        "python -m pip install -q 'torch-scatter==2.0.9' 'torch-sparse==0.6.13' "
        "'torch-cluster==1.5.9' 'torch-spline-conv==1.2.1' "
        "-f https://data.pyg.org/whl/torch-1.10.1+cu113.html"
    )
    mex('metalbind',"python -m pip install -q 'torch-geometric==2.0.4' 'pykeops==2.1'")

    pymesh_wheel=CACHE/'pymesh2-0.3-cp37-cp37m-linux_x86_64.whl'
    if not pymesh_wheel.exists():
        sh(
            'curl -L --fail --retry 3 '
            '"https://sourceforge.net/projects/pymesh.mirror/files/v0.3/'
            'pymesh2-0.3-cp37-cp37m-linux_x86_64.whl/download" '
            f'-o "{pymesh_wheel}"'
        )
    mex('metalbind',f'python -m pip install -q "{pymesh_wheel}"')

    envbin=LOCAL_MAMBA_ROOT/'envs'/'metalbind'/'bin'
    reduce_bin=envbin/'reduce'
    msms_bin=envbin/'msms'
    if not reduce_bin.exists() or not msms_bin.exists():
        raise RuntimeError(f'MetalBind binaries missing: reduce={reduce_bin.exists()}, msms={msms_bin.exists()}')

    (METALB/'default_config/bin_path.py').write_text(
        "bin_path = {}\n"
        f"bin_path['REDUCE'] = r'{reduce_bin}'\n"
        f"bin_path['MSMS'] = r'{msms_bin}'\n"
    )

    rr=metal_mex(
        r'''python - <<'INNERPY'
import torch, pymesh, torch_scatter, torch_sparse, torch_cluster, torch_geometric, pykeops
from torch_sparse import SparseTensor
print('torch',torch.__version__,'torch CUDA',torch.version.cuda,'cuda',torch.cuda.is_available())
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('pyg',torch_geometric.__version__,'pykeops',pykeops.__version__)
assert torch.cuda.is_available()
row=torch.tensor([0,1],dtype=torch.long,device='cuda')
col=torch.tensor([1,0],dtype=torch.long,device='cuda')
A=SparseTensor(row=row,col=col,value=torch.ones(2,device='cuda'),sparse_sizes=(2,2))
y=A.matmul(torch.ones((2,1),device='cuda'))
print('torch_sparse CUDA op',y.flatten().tolist())
print('METALBIND GPU STACK: PASS')
INNERPY''',
        check=False,capture=True
    )
    print(rr.stdout)
    if rr.returncode!=0:
        raise RuntimeError(f'MetalBind GPU stack failed: return code {rr.returncode}')
else:
    print('MetalBind runtime setup: SKIP (strict final prediction cache reuse).')


MetalBind: all 13 ions x Test300/Test46 strictly cached -> preprocessing + inference can be skipped.
MetalBind runtime setup: SKIP (strict final prediction cache reuse).


In [20]:
# Reconnect persistent MetalBind preprocessing cache on Drive

METALB_PREP_ROOT = (
    WORK / 'metalbind_external_v6' / 'prepared'
)

for dataset, df in DATASETS.items():
    root = METALB_PREP_ROOT / dataset

    assert (root / 'manifest.csv').exists(), (
        dataset,
        'manifest missing'
    )

    missing = []

    for r in df.itertuples():
        pair = f'{r.prot_id}:A:X1'

        if not (root / 'data_label' / pair).exists():
            missing.append((r.prot_id, 'pickle'))

        if not (
            root
            / 'data_label'
            / f'{pair}.surface_map.json'
        ).exists():
            missing.append((r.prot_id, 'map'))

    if missing:
        raise RuntimeError(
            f'{dataset}: missing preprocessing cache: '
            f'{missing[:10]}'
        )

    print(
        dataset,
        f'18D CACHE OK: {len(df)}/{len(df)}'
    )

TestSet300 18D CACHE OK: 300/300
TestSetNew46 18D CACHE OK: 46/46


## 7. ESM-C 600M features — persistent cache first


In [21]:
# ============================================================
# Persistent ESM-C preflight
# ============================================================
def _metalb_esmc_cache_complete():
    if METALBIND_FINAL_CACHE_COMPLETE:
        return True
    if REBUILD_METALBIND_ESMC:
        return False
    for dataset,df in DATASETS.items():
        outdir=CACHE/'metalbind_esmc600m_v6'/dataset
        for r in df.itertuples():
            fp=outdir/f'{r.prot_id}.npy'
            if not fp.exists():
                return False
            try:
                a=np.load(fp,mmap_mode='r')
            except Exception:
                return False
            if a.shape!=(len(r.sequence),1152) or not np.all(np.isfinite(a)):
                return False
    return True

METALB_ESMC_CACHE_COMPLETE=_metalb_esmc_cache_complete()
METALB_ESMC_NEED_GENERATION=(RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE and (REBUILD_METALBIND_ESMC or not METALB_ESMC_CACHE_COMPLETE))
if METALB_ESMC_CACHE_COMPLETE and RUN_METALBIND_TRANSFER:
    print('MetalBind ESM-C: complete persistent 1152-d cache found -> ESM-C env/model load SKIP')

# ============================================================
# 18B. MetalBind ESM-C 600M / 1152-d FEATURES — FINAL
# ============================================================

BIOHUB_TRANSFORMERS_COMMIT = (
    "3a8956fb4d4ea16b0ec8e71deef2c2909b6a5cbf"
)

METALB_ESMC_HF_MODEL = "biohub/ESMC-600M"

METALB_ESMC_ROOT = CACHE / "metalbind_esmc600m_v6"
METALB_ESMC_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 18B-1. ESM-C ISOLATED ENVIRONMENT
# ============================================================

if METALB_ESMC_NEED_GENERATION:

    create_env(
        "esmc",
        "3.10"
    )

    mex(
        "esmc",
        "python -m pip install -q --upgrade "
        "'pip<25' setuptools wheel"
    )

    # Base numerical / GPU environment
    mex(
        "esmc",
        "python -m pip install -q "
        "'torch==2.4.1' "
        "'numpy<2' "
        "biopython "
        "safetensors"
    )

    # ESM-C support is in Biohub's Transformers fork.
    rr = mex(
        "esmc",
        (
            "python -m pip install -q --upgrade "
            f"'git+https://github.com/Biohub/transformers.git@"
            f"{BIOHUB_TRANSFORMERS_COMMIT}' "
            "'huggingface-hub>=0.34,<1' "
            "'tokenizers>=0.22,<=0.23' "
            "'safetensors>=0.4.3'"
        ),
        check=False,
        capture=True
    )

    if rr.stdout:
        print(rr.stdout)

    if getattr(rr, "stderr", None):
        print(rr.stderr)

    if rr.returncode != 0:
        raise RuntimeError(
            "Biohub Transformers installation failed: "
            f"{rr.returncode}"
        )


    # --------------------------------------------------------
    # Optional HuggingFace token
    # --------------------------------------------------------

    if not (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_TOKEN")
    ):
        try:
            from google.colab import userdata

            _hf = userdata.get("HF_TOKEN")

            if _hf:
                os.environ["HF_TOKEN"] = _hf
                print(
                    "HF_TOKEN loaded from Colab Secrets."
                )

        except Exception:
            pass


    # --------------------------------------------------------
    # Strict environment smoke test
    # --------------------------------------------------------

    rr = mex(
        "esmc",
        r"""python - <<'PY'
import torch
import transformers

from transformers.models.esmc.tokenization_esmc import ESMCTokenizer
from transformers.models.esmc.modeling_esmc import ESMCModel

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available()

print("ESMCTokenizer import: PASS")
print("ESMCModel import: PASS")
print("ESM-C environment: PASS")
PY""",
        check=False,
        capture=True
    )

    print(rr.stdout)

    if rr.returncode != 0:
        raise RuntimeError(
            "ESM-C environment smoke test failed: "
            f"{rr.returncode}"
        )


# ============================================================
# 18B-2. WRITE PER-DATASET ESM-C GENERATION SCRIPT
# ============================================================

def write_esmc_generation_script(
    dataset,
    df,
    outdir
):

    fasta = (
        CANON
        /
        f"{dataset}.fasta"
    )

    script = (
        WORK
        /
        f"generate_metalbind_esmc_{dataset}.py"
    )

    script.write_text(
f"""
import os
from pathlib import Path

import numpy as np
import torch

from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel


MODEL_ID = {METALB_ESMC_HF_MODEL!r}

FASTA = Path(r"{fasta}")
OUT = Path(r"{outdir}")

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# GPU guard
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "MetalBind ESM-C extraction requires CUDA"
    )


# ------------------------------------------------------------
# Optional HuggingFace authentication
# ------------------------------------------------------------

token = (
    os.environ.get("HF_TOKEN")
    or
    os.environ.get("HUGGINGFACE_TOKEN")
)

kwargs = (
    {{"token": token}}
    if token
    else {{}}
)


# ------------------------------------------------------------
# Official HF-compatible Biohub ESM-C 600M
# ------------------------------------------------------------

print(
    "Loading ESM-C:",
    MODEL_ID
)

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    **kwargs
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    **kwargs
).to(
    "cuda"
).eval()


print(
    "ESM-C d_model:",
    model.config.d_model
)

if model.config.d_model != 1152:
    raise RuntimeError(
        "Unexpected ESM-C hidden dimension: "
        f"{{model.config.d_model}}"
    )


# ------------------------------------------------------------
# Generate one residue-level embedding per protein
# ------------------------------------------------------------

records = list(
    SeqIO.parse(
        str(FASTA),
        "fasta"
    )
)

print(
    "Proteins:",
    len(records)
)


for idx, rec in enumerate(
    records,
    start=1
):

    pid = str(rec.id)
    seq = str(rec.seq)

    fp = (
        OUT
        /
        f"{{pid}}.npy"
    )


    # --------------------------------------------------------
    # Cache reuse
    # --------------------------------------------------------

    if fp.exists():

        try:

            a = np.load(
                fp,
                mmap_mode="r"
            )

            if (
                a.shape
                ==
                (
                    len(seq),
                    1152
                )
                and
                np.all(
                    np.isfinite(a)
                )
            ):
                print(
                    f"[{{idx}}/{{len(records)}}] "
                    f"{{pid}} cached"
                )

                continue

        except Exception:
            pass


    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    enc = tok(
        seq,
        return_tensors="pt",
        add_special_tokens=True,
        return_special_tokens_mask=True,
        truncation=False,
    )

    special = enc.pop(
        "special_tokens_mask"
    )[0].bool()


    enc = {{
        k: v.to("cuda")
        for k, v
        in enc.items()
    }}


    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    with torch.inference_mode():

        out = model(
            **enc,
            return_dict=True
        )


    if (
        not hasattr(
            out,
            "last_hidden_state"
        )
        or
        out.last_hidden_state
        is None
    ):
        raise RuntimeError(
            f"{{pid}}: "
            "ESMCModel output missing "
            "last_hidden_state"
        )


    h = (
        out
        .last_hidden_state[0]
        .float()
        .cpu()
    )


    # --------------------------------------------------------
    # Remove CLS/EOS special tokens
    # --------------------------------------------------------

    special_cpu = (
        special
        .cpu()
    )


    if (
        special_cpu.shape[0]
        !=
        h.shape[0]
    ):
        raise RuntimeError(
            f"{{pid}}: "
            f"special-token mask length "
            f"{{special_cpu.shape[0]}} "
            f"!= hidden length "
            f"{{h.shape[0]}}"
        )


    h = h[
        ~special_cpu
    ]


    arr = (
        h.numpy()
        .astype(
            np.float32,
            copy=False
        )
    )


    # --------------------------------------------------------
    # STRICT residue alignment
    # --------------------------------------------------------

    expected = (
        len(seq),
        1152
    )


    if arr.shape != expected:
        raise RuntimeError(
            f"{{pid}}: "
            f"ESM-C shape "
            f"{{arr.shape}}, "
            f"expected "
            f"{{expected}}"
        )


    if not np.all(
        np.isfinite(arr)
    ):
        raise RuntimeError(
            f"{{pid}}: "
            "non-finite ESM-C embedding"
        )


    np.save(
        fp,
        arr
    )


    print(
        f"[{{idx}}/{{len(records)}}] "
        f"{{pid}} PASS "
        f"{{arr.shape}}"
    )


print(
    "ESM-C generation complete:",
    len(
        list(
            OUT.glob("*.npy")
        )
    )
)
"""
    )

    return script


# ============================================================
# 18B-3. GENERATE + STRICT AUDIT
# ============================================================

if RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE:

    for dataset, df in DATASETS.items():

        print(
            "\n"
            +
            "=" * 72
        )

        print(
            "MetalBind ESM-C:",
            dataset
        )

        print(
            "=" * 72
        )


        outdir = (
            METALB_ESMC_ROOT
            /
            dataset
        )

        outdir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ----------------------------------------------------
        # Check whether generation is needed
        # ----------------------------------------------------

        need = (
            REBUILD_METALBIND_ESMC
        )


        if not need:

            for r in df.itertuples():

                fp = (
                    outdir
                    /
                    f"{r.prot_id}.npy"
                )


                if not fp.exists():

                    need = True
                    break


                try:

                    a = np.load(
                        fp,
                        mmap_mode="r"
                    )

                except Exception:

                    need = True
                    break


                if (
                    a.shape
                    !=
                    (
                        len(r.sequence),
                        1152
                    )
                    or
                    not np.all(
                        np.isfinite(a)
                    )
                ):

                    need = True
                    break


        # ----------------------------------------------------
        # Generate missing/invalid embeddings
        # ----------------------------------------------------

        if need:

            script = (
                write_esmc_generation_script(
                    dataset,
                    df,
                    outdir
                )
            )


            print(
                "Generation script:",
                script
            )


            rr = mex(
                "esmc",
                f'python "{script}"',
                check=False,
                capture=True
            )


            if rr.stdout:
                print(
                    rr.stdout
                )


            if getattr(
                rr,
                "stderr",
                None
            ):
                print(
                    rr.stderr
                )


            if rr.returncode != 0:

                raise RuntimeError(
                    f"{dataset}: "
                    "ESM-C generation failed "
                    f"with return code "
                    f"{rr.returncode}. "
                    "Full subprocess log "
                    "printed above."
                )


        # ----------------------------------------------------
        # FINAL FULL-DATASET AUDIT
        # ----------------------------------------------------

        bad = []


        for r in df.itertuples():

            pid = str(
                r.prot_id
            )

            fp = (
                outdir
                /
                f"{pid}.npy"
            )


            if not fp.exists():

                bad.append(
                    (
                        pid,
                        "missing"
                    )
                )

                continue


            try:

                a = np.load(
                    fp,
                    mmap_mode="r"
                )

            except Exception as e:

                bad.append(
                    (
                        pid,
                        f"unreadable: {e}"
                    )
                )

                continue


            expected = (
                len(r.sequence),
                1152
            )


            if a.shape != expected:

                bad.append(
                    (
                        pid,
                        f"shape={a.shape}, "
                        f"expected={expected}"
                    )
                )


            elif not np.all(
                np.isfinite(a)
            ):

                bad.append(
                    (
                        pid,
                        "non-finite"
                    )
                )


        if bad:

            raise RuntimeError(
                f"{dataset}: "
                "ESM-C final audit failed: "
                f"{bad[:10]}"
            )


        print(
            f"{dataset}: "
            f"ESM-C audit PASS "
            f"{len(df)}/{len(df)} proteins."
        )


    print(
        "\n"
        "MetalBind ESM-C 600M audit "
        "PASSED for TestSet300 + TestSetNew46."
    )


MetalBind ESM-C: complete persistent 1152-d cache found -> ESM-C env/model load SKIP


## 8. Author-format manifests


In [ ]:
# ============================================================
# 18C. CONVERT ESM-C TO AUTHOR TEXT FORMAT + PREPARE MANIFESTS
# ============================================================
METALB_PREP_ROOT=METALB_EXT/'prepared'
METALB_PREP_ROOT.mkdir(parents=True,exist_ok=True)

if RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE:
    for dataset,df in DATASETS.items():
        root=METALB_PREP_ROOT/dataset
        raw_pdb=root/'raw_pdb'
        esm_txt=root/'600M'
        raw_pdb.mkdir(parents=True,exist_ok=True)
        esm_txt.mkdir(parents=True,exist_ok=True)

        manifest_rows=[]
        for r in df.itertuples():
            pid=str(r.prot_id)
            if ':' in pid or '/' in pid:
                raise RuntimeError(f'MetalBind external adapter requires path-safe protein id, got {pid}')

            # Author extract_multimer/get_seq expects <pdb_id><chain>.pdb, hence pidA.pdb.
            pdb_dst=raw_pdb/f'{pid}A.pdb'
            shutil.copy2(NORMALIZED_EXTERNAL_PDB[dataset][pid],pdb_dst)

            txt=esm_txt/f'{pid}A.txt'
            npy=METALB_ESMC_ROOT/dataset/f'{pid}.npy'
            if REBUILD_METALBIND_ESMC or not txt.exists():
                arr=np.load(npy)
                np.savetxt(txt,arr,fmt='%.10e')

            # Round-trip author loader format check.
            aa=np.loadtxt(txt,dtype=np.float32)
            if aa.shape!=(len(r.sequence),1152):
                raise RuntimeError(f'{dataset}/{pid}: author ESM text shape {aa.shape}')

            manifest_rows.append([pid,r.sequence,str(pdb_dst),str(txt)])

        man=pd.DataFrame(manifest_rows,columns=['prot_id','sequence','pdb_path','esm_txt'])
        man.to_csv(root/'manifest.csv',index=False)
        print(dataset,'MetalBind manifest',len(man))
if RUN_METALBIND_TRANSFER and METALBIND_FINAL_CACHE_COMPLETE:
    print('18C manifests: SKIP (final MetalBind ion predictions already complete).')


TestSet300 MetalBind manifest 300
TestSetNew46 MetalBind manifest 46


## 9. MSMS short-path + KeOps/NVRTC compatibility


In [ ]:
# ============================================================
# 18D-0. MetalBind RUNTIME COMPATIBILITY PATCHES — BEFORE PROCESSOR
# A) legacy MSMS short local path; B) exact-formula KeOps GPU smoke + CPU fallback
# ============================================================
if RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE:
    from pathlib import Path
    import textwrap

    APPLY_MSMS = METALB / 'compute_surface' / 'apply_msms.py'

    if not APPLY_MSMS.exists():
        raise FileNotFoundError(
            f'MetalBind apply_msms.py not found: {APPLY_MSMS}. Run 18A first.'
        )

    APPLY_MSMS.write_text(textwrap.dedent(r'''
    from default_config.bin_path import bin_path
    from compute_surface.read_msms import read_msms

    from subprocess import Popen, PIPE
    from pathlib import Path

    import shutil
    import tempfile


    def computeMSMS(pair, dir_opts, probe_radius):

        # --------------------------------------------------------
        # Author-generated XYZRN input
        # --------------------------------------------------------
        source_xyzrn = (
            Path(dir_opts['xyzrn_dir'])
            / f'{pair}.xyzrn'
        )

        if not source_xyzrn.exists():
            raise FileNotFoundError(
                f'MSMS input XYZRN missing: {source_xyzrn}'
            )

        if source_xyzrn.stat().st_size == 0:
            raise RuntimeError(
                f'MSMS input XYZRN is empty: {source_xyzrn}'
            )

        # --------------------------------------------------------
        # SHORT LOCAL PATHS ONLY for the native legacy MSMS binary
        # --------------------------------------------------------
        content_dir = Path('/content')
        scratch_parent = (
            content_dir
            if content_dir.is_dir()
            else Path(tempfile.gettempdir())
        )

        scratch = Path(
            tempfile.mkdtemp(
                prefix='mbmsms_',
                dir=str(scratch_parent),
            )
        )

        local_xyzrn = scratch / 'in.xyzrn'
        output_base = scratch / 'surface'

        try:
            shutil.copy2(
                source_xyzrn,
                local_xyzrn,
            )

            msms_bin = str(bin_path['MSMS'])

            args = [
                msms_bin,
                '-density', '3',
                '-hdensity', '3',
                '-probe', str(probe_radius),
                '-if', str(local_xyzrn),
                '-of', str(output_base),
                '-af', str(output_base),
            ]

            print(
                'MSMS COMMAND:',
                ' '.join(args),
            )

            p = Popen(
                args,
                stdout=PIPE,
                stderr=PIPE,
            )

            # IMPORTANT: this must be one assignment.
            stdout, stderr = p.communicate()

            stdout_text = stdout.decode(
                'utf-8',
                errors='replace',
            )
            stderr_text = stderr.decode(
                'utf-8',
                errors='replace',
            )

            if p.returncode != 0:
                raise RuntimeError(
                    'MSMS failed.\n'
                    f'Return code: {p.returncode}\n'
                    f'Input: {local_xyzrn}\n'
                    f'STDOUT:\n{stdout_text}\n'
                    f'STDERR:\n{stderr_text}'
                )

            vert = Path(str(output_base) + '.vert')
            face = Path(str(output_base) + '.face')

            for fp in (vert, face):
                if not fp.exists():
                    raise RuntimeError(
                        'MSMS returned success but output is missing: '
                        f'{fp}\n'
                        f'STDOUT:\n{stdout_text}\n'
                        f'STDERR:\n{stderr_text}'
                    )
                if fp.stat().st_size == 0:
                    raise RuntimeError(
                        f'MSMS output is empty: {fp}'
                    )

            # Legacy MSMS .vert/.face are supposed to be ASCII text.
            for fp in (vert, face):
                raw = fp.read_bytes()
                try:
                    raw.decode('ascii')
                except UnicodeDecodeError as e:
                    raise RuntimeError(
                        'MSMS generated non-ASCII output despite '
                        'short-path execution.\n'
                        f'File: {fp}\n'
                        f'Size: {len(raw)} bytes\n'
                        f'First 256 bytes: {raw[:256]!r}\n'
                        f'Decode error: {e}\n'
                        f'MSMS STDOUT:\n{stdout_text}\n'
                        f'MSMS STDERR:\n{stderr_text}'
                    )

            # AUTHORS' original parser.
            vertices, faces, normalv, res_id = read_msms(
                str(output_base)
            )

            # Strict geometry checks.
            if getattr(vertices, 'ndim', None) != 2 or vertices.shape[1] != 3:
                raise RuntimeError(
                    f'MSMS vertices must be N x 3, got {getattr(vertices, "shape", None)}'
                )

            if getattr(faces, 'ndim', None) != 2 or faces.shape[1] != 3:
                raise RuntimeError(
                    f'MSMS faces must be M x 3, got {getattr(faces, "shape", None)}'
                )

            if normalv.shape != vertices.shape:
                raise RuntimeError(
                    'MSMS normal/vertex shape mismatch: '
                    f'{normalv.shape} vs {vertices.shape}'
                )

            if len(res_id) != len(vertices):
                raise RuntimeError(
                    'MSMS residue-ID/vertex count mismatch: '
                    f'{len(res_id)} vs {len(vertices)}'
                )

            if len(vertices) == 0 or len(faces) == 0:
                raise RuntimeError(
                    'MSMS produced empty geometry'
                )

            print(
                'MSMS PASS:',
                f'vertices={len(vertices)}',
                f'faces={len(faces)}',
            )

            return (
                vertices,
                faces,
                normalv,
                res_id,
            )

        finally:
            shutil.rmtree(
                scratch,
                ignore_errors=True,
            )
    '''))

    print('Patched short-path MSMS:', APPLY_MSMS)



    rr=metal_preprocess_mex(
        f'python -m py_compile "{APPLY_MSMS}"',
        cwd=METALB,
        check=False,
        capture=True
    )
    if rr.stdout:
        print(rr.stdout)
    if rr.returncode!=0:
        raise RuntimeError('Patched apply_msms.py failed Python compilation')

    def _keops_smoke(backend):
        script = r'''import numpy as np
from pykeops.numpy import LazyTensor
np.random.seed(1)
N=32
points=np.random.randn(N,3).astype(np.float64)/6.0
normals=np.random.randn(N,3).astype(np.float64)
normals/=np.linalg.norm(normals,axis=1,keepdims=True)
x,y,z=normals[...,0],normals[...,1],normals[...,2]
s=(2*(z>=0))-1.0
a=-1/(s+z)
b=x*y*a
uv=np.stack((1+s*x*x*a,s*b,-s*x,b,s+y*y*a,-y),axis=-1)
tangent_bases=uv.reshape((N,2,3))
weights=np.random.rand(N,1).astype(np.float64)
weights_j=LazyTensor(weights[:,0].reshape(1,-1,1))
x_i=LazyTensor(points[:,None,:])
x_j=LazyTensor(points[None,:,:])
n_i=LazyTensor(normals[:,None,:])
n_j=LazyTensor(normals[None,:,:])
uv_i=LazyTensor(tangent_bases.reshape([-1,1,6]))
rho2_ij=((x_j-x_i)**2).sum(-1)*((2-(n_i|n_j))**2)
alpha=3.0
window_ij=(1.0+rho2_ij/alpha)**(-alpha)
X_ij=uv_i.matvecmult(x_j-x_i)
orientation_vector_ij=(window_ij*weights_j)*X_ij
out=orientation_vector_ij.sum(dim=1,backend=BACKEND_TOKEN)
assert out.shape==(N,2)
assert np.all(np.isfinite(out))
print('KEOPS BACKEND_TOKEN EXACT-FORMULA SMOKE: PASS',out.shape)
'''.replace('BACKEND_TOKEN',repr(backend))
        smoke=WORK/f'metalbind_keops_smoke_{backend.lower()}.py'
        smoke.write_text(script)
        return metal_preprocess_mex(
            f'python "{smoke}"',
            cwd=METALB,
            check=False,
            capture=True
        )

    requested=str(METALBIND_KEOPS_MODE).upper()
    if requested not in {'AUTO','GPU','CPU'}:
        raise ValueError("METALBIND_KEOPS_MODE must be 'auto', 'GPU', or 'CPU'")

    gpu_rr=None
    if requested in {'AUTO','GPU'}:
        gpu_rr=_keops_smoke('GPU')
        print(gpu_rr.stdout or '')

    if requested=='GPU' and gpu_rr.returncode!=0:
        raise RuntimeError(
            'METALBIND_KEOPS_MODE=GPU but the exact author KeOps formula failed GPU/NVRTC compilation. '
            'Use auto or CPU on this runtime.'
        )

    if requested=='CPU' or (requested=='AUTO' and gpu_rr.returncode!=0):
        cpu_rr=_keops_smoke('CPU')
        print(cpu_rr.stdout or '')
        if cpu_rr.returncode!=0:
            raise RuntimeError('KeOps CPU fallback failed to compile the exact MetalBind tangent formula.')
        selected='CPU'
    else:
        selected='GPU'

    mbp=METALB/'MetalBindProcessor.py'
    text=mbp.read_text()
    author_line='orientation_vector_i = orientation_vector_ij.sum(dim=1)'
    cpu_line='orientation_vector_i = orientation_vector_ij.sum(dim=1, backend="CPU")'
    gpu_line='orientation_vector_i = orientation_vector_ij.sum(dim=1, backend="GPU")'
    text=text.replace(cpu_line,author_line).replace(gpu_line,author_line)
    if author_line not in text:
        raise RuntimeError('Could not locate the author KeOps tangent reduction line in MetalBindProcessor.py')
    if selected=='CPU':
        text=text.replace(author_line,cpu_line,1)
    else:
        text=text.replace(author_line,gpu_line,1)
    mbp.write_text(text)

    METALBIND_KEOPS_SELECTED=selected
    (MANIFEST/'metalbind_keops_runtime.json').write_text(json.dumps({
        'requested':METALBIND_KEOPS_MODE,
        'selected':selected,
        'gpu_smoke_returncode':None if gpu_rr is None else int(gpu_rr.returncode),
        'note':'Only the KeOps reduction execution backend changed; tangent-frame formula unchanged.'
    },indent=2))
    print('MetalBind KeOps selected backend:',selected)
else:
    METALBIND_KEOPS_SELECTED='CACHED_FINAL' if METALBIND_FINAL_CACHE_COMPLETE else 'NOT_RUN'
    print('18D-0 compatibility patches: SKIP')


Patched short-path MSMS: /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/repos/metalbind/compute_surface/apply_msms.py
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python -m py_compile "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/repos/metalbind/compute_surface/apply_msms.py"
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_keops_smoke_gpu.py"
  File "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_keops_smoke_gpu.py", line 29
    print('KEOPS 'GPU' EXACT-FORMULA SMOKE: PASS',out.shape)
                    ^
SyntaxError: invalid syntax

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_keops_

RuntimeError: KeOps CPU fallback failed to compile the exact MetalBind tangent formula.

In [ ]:
# ============================================================
# HOTFIX ONLY — KeOps smoke-test quoting bug
# Run this immediately after the failed 18D-0 cell.
# No restart. No cache deletion. No rebuild.
# ============================================================

def _keops_smoke(backend):
    script = r'''import numpy as np
from pykeops.numpy import LazyTensor

BACKEND = BACKEND_TOKEN

np.random.seed(1)
N = 32

points = np.random.randn(N, 3).astype(np.float64) / 6.0

normals = np.random.randn(N, 3).astype(np.float64)
normals /= np.linalg.norm(normals, axis=1, keepdims=True)

x, y, z = normals[..., 0], normals[..., 1], normals[..., 2]

s = (2 * (z >= 0)) - 1.0
a = -1 / (s + z)
b = x * y * a

uv = np.stack(
    (
        1 + s * x * x * a,
        s * b,
        -s * x,
        b,
        s + y * y * a,
        -y,
    ),
    axis=-1,
)

tangent_bases = uv.reshape((N, 2, 3))

weights = np.random.rand(N, 1).astype(np.float64)

weights_j = LazyTensor(
    weights[:, 0].reshape(1, -1, 1)
)

x_i = LazyTensor(
    points[:, None, :]
)

x_j = LazyTensor(
    points[None, :, :]
)

n_i = LazyTensor(
    normals[:, None, :]
)

n_j = LazyTensor(
    normals[None, :, :]
)

uv_i = LazyTensor(
    tangent_bases.reshape([-1, 1, 6])
)

rho2_ij = (
    ((x_j - x_i) ** 2).sum(-1)
    *
    ((2 - (n_i | n_j)) ** 2)
)

alpha = 3.0

window_ij = (
    1.0 + rho2_ij / alpha
) ** (-alpha)

X_ij = uv_i.matvecmult(
    x_j - x_i
)

orientation_vector_ij = (
    window_ij * weights_j
) * X_ij

out = orientation_vector_ij.sum(
    dim=1,
    backend=BACKEND
)

assert out.shape == (N, 2)
assert np.all(np.isfinite(out))

print(
    "KEOPS",
    BACKEND,
    "EXACT-FORMULA SMOKE: PASS",
    out.shape
)
'''.replace(
        'BACKEND_TOKEN',
        repr(backend)
    )

    smoke = (
        WORK
        / f'metalbind_keops_smoke_{backend.lower()}.py'
    )

    smoke.write_text(script)

    # First make sure THIS GENERATED SCRIPT is valid Python.
    compile_rr = metal_preprocess_mex(
        f'python -m py_compile "{smoke}"',
        cwd=METALB,
        check=False,
        capture=True
    )

    if compile_rr.returncode != 0:
        print(compile_rr.stdout or '')
        raise RuntimeError(
            f'Generated {backend} KeOps smoke script '
            'has invalid Python syntax.'
        )

    return metal_preprocess_mex(
        f'python "{smoke}"',
        cwd=METALB,
        check=False,
        capture=True
    )


# ============================================================
# Re-run ONLY KeOps backend selection
# ============================================================

requested = str(
    METALBIND_KEOPS_MODE
).upper()

if requested not in {
    'AUTO',
    'GPU',
    'CPU'
}:
    raise ValueError(
        "METALBIND_KEOPS_MODE must be "
        "'auto', 'GPU', or 'CPU'"
    )


gpu_rr = None

if requested in {
    'AUTO',
    'GPU'
}:
    print(
        "\nTesting MetalBind KeOps GPU backend..."
    )

    gpu_rr = _keops_smoke(
        'GPU'
    )

    print(
        gpu_rr.stdout or ''
    )


if (
    requested == 'GPU'
    and gpu_rr.returncode != 0
):
    raise RuntimeError(
        'METALBIND_KEOPS_MODE=GPU but '
        'the exact MetalBind KeOps formula '
        'failed GPU/NVRTC compilation. '
        'Use AUTO or CPU on this runtime.'
    )


if (
    requested == 'CPU'
    or (
        requested == 'AUTO'
        and gpu_rr.returncode != 0
    )
):
    print(
        "\nGPU unavailable/incompatible; "
        "testing exact formula on KeOps CPU..."
    )

    cpu_rr = _keops_smoke(
        'CPU'
    )

    print(
        cpu_rr.stdout or ''
    )

    if cpu_rr.returncode != 0:
        raise RuntimeError(
            'KeOps CPU fallback genuinely failed '
            'to compile the exact MetalBind '
            'tangent formula.'
        )

    selected = 'CPU'

else:
    selected = 'GPU'


# ============================================================
# Patch ONLY reduction backend in author's tangent formula
# Formula itself remains unchanged.
# ============================================================

mbp = (
    METALB
    / 'MetalBindProcessor.py'
)

text = mbp.read_text()

author_line = (
    'orientation_vector_i = '
    'orientation_vector_ij.sum(dim=1)'
)

cpu_line = (
    'orientation_vector_i = '
    'orientation_vector_ij.sum('
    'dim=1, backend="CPU")'
)

gpu_line = (
    'orientation_vector_i = '
    'orientation_vector_ij.sum('
    'dim=1, backend="GPU")'
)

# Normalize any previous backend patch.
text = (
    text
    .replace(
        cpu_line,
        author_line
    )
    .replace(
        gpu_line,
        author_line
    )
)

if author_line not in text:
    raise RuntimeError(
        'Could not locate the author KeOps '
        'tangent reduction line in '
        'MetalBindProcessor.py'
    )

if selected == 'CPU':
    text = text.replace(
        author_line,
        cpu_line,
        1
    )
else:
    text = text.replace(
        author_line,
        gpu_line,
        1
    )

mbp.write_text(text)

METALBIND_KEOPS_SELECTED = selected

(
    MANIFEST
    / 'metalbind_keops_runtime.json'
).write_text(
    json.dumps(
        {
            'requested':
                METALBIND_KEOPS_MODE,

            'selected':
                selected,

            'gpu_smoke_returncode':
                None
                if gpu_rr is None
                else int(
                    gpu_rr.returncode
                ),

            'note':
                'Only the KeOps reduction '
                'execution backend changed; '
                'tangent-frame formula unchanged.'
        },
        indent=2
    )
)

print(
    "\n============================================"
)

print(
    "MetalBind KeOps selected backend:",
    selected
)

print(
    "============================================"
)


Testing MetalBind KeOps GPU backend...
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python -m py_compile "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_keops_smoke_gpu.py"
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_keops_smoke_gpu.py"
[KeOps] Generating code for formula Sum_Reduction((Powf(Var(5,1,2)+(Sum((Var(0,3,1)-Var(1,3,0))**2)*(2-Var(2,3,0)|Var(3,3,1))**2)/Var(4,1,2),Var(6,1,2))*Var(7,1,1))*MatVecMult(Var(8,6,0),Var(0,3,1)-Var(1,3,0)),0) ... OK
KEOPS GPU EXACT-FORMULA SMOKE: PASS (32, 2)


MetalBind KeOps selected backend: GPU


## 10. Surface preprocessing — resume per protein


In [ ]:
# ============================================================
# 18D. WRITE + RUN LABEL-FREE ADAPTER AROUND AUTHORS' MetalBindProcessor
# FINAL RESUME-SAFE VERSION
# ============================================================

METALB_PREP_SCRIPT = METALB / 'external_prepare_v6.py'

METALB_PREP_SCRIPT.write_text(r'''
import argparse
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

from MetalBindProcessor import MetalBindProcessor


# ============================================================
# Label-free inference adapters
# ============================================================

def dummy_nucleic(self):
    return []


def dummy_interface(self, anno):
    # Inference only: labels/interfaces are placeholders and
    # are not used to generate predictions.
    return np.zeros(
        self.mesh.vertices.shape[0],
        dtype=np.float32
    )


def dummy_site(self, anno):
    # Preserve exact surface-residue ordering from author's processor.
    s2i = {
        res: i
        for i, res in enumerate(self.surface_res)
    }

    point_res_id = np.zeros(
        len(self.vert_info),
        dtype=np.int64
    )

    for k, info in enumerate(self.vert_info):
        fields = info.split('_')

        if len(fields) < 2:
            raise RuntimeError(
                f'Unexpected vert_info entry: {info}'
            )

        rid = fields[0] + '_' + fields[1]

        if rid not in s2i:
            raise RuntimeError(
                f'point residue {rid} missing from surface-residue map'
            )

        point_res_id[k] = s2i[rid]

    self.site_by_biolip = set()

    return (
        np.zeros(
            len(self.surface_res),
            dtype=np.int64
        ),
        point_res_id
    )


# Patch labels only; structural preprocessing remains author's code.
MetalBindProcessor._get_nucleic_acid_atom = dummy_nucleic
MetalBindProcessor._get_interface_mental = dummy_interface
MetalBindProcessor._get_site_biolip = dummy_site


# ============================================================
# CLI
# ============================================================

ap = argparse.ArgumentParser()
ap.add_argument('--manifest', required=True)
ap.add_argument('--root', required=True)
ap.add_argument('--rebuild', action='store_true')
a = ap.parse_args()

root = Path(a.root)
manifest_path = Path(a.manifest)

if not manifest_path.exists():
    raise FileNotFoundError(
        f'Manifest not found: {manifest_path}'
    )

manifest = pd.read_csv(manifest_path)

if len(manifest) == 0:
    raise RuntimeError(
        f'Empty manifest: {manifest_path}'
    )

required_cols = {
    'prot_id',
    'sequence',
    'pdb_path',
    'esm_txt',
}

missing_cols = required_cols - set(manifest.columns)

if missing_cols:
    raise RuntimeError(
        f'Manifest missing columns: {sorted(missing_cols)}'
    )


# ============================================================
# Output directories
# ============================================================

for d in [
    'chain_pdb',
    'protonated_pdb',
    'xyzrn',
    'msms',
    'data_label',
    'ligand_dummy',
    'hmm',
    'fasta',
    'dssp',
    'save',
    'mesh',
    'mesh_fix',
    'ply',
]:
    (root / d).mkdir(
        parents=True,
        exist_ok=True
    )


raw_pdb = Path(
    manifest.iloc[0].pdb_path
).parent

esm_dir = Path(
    manifest.iloc[0].esm_txt
).parent


if not raw_pdb.exists():
    raise FileNotFoundError(
        f'raw_pdb directory not found: {raw_pdb}'
    )

if not esm_dir.exists():
    raise FileNotFoundError(
        f'ESM directory not found: {esm_dir}'
    )


opts = {
    'base_dir': str(root),
    'raw_pdb_dir': str(raw_pdb),
    'ligand_dir': str(root / 'ligand_dummy'),
    'hhm_dir': str(root / 'hmm'),
    'fasta_dir': str(root / 'fasta'),
    'dssp': str(root / 'dssp'),
    'protonated_pdb_dir': str(root / 'protonated_pdb'),
    'chain_pdb_dir': str(root / 'chain_pdb'),
    'xyzrn_dir': str(root / 'xyzrn'),
    'msms_dir': str(root / 'msms'),
    'data_label': str(root / 'data_label'),
    'save_dir': str(root / 'save'),
    'mesh_dir': str(root / 'mesh'),
    'mesh_fix_dir': str(root / 'mesh_fix'),
    'ply_dir': str(root / 'ply'),
    'esm_dir': str(esm_dir),
}


print(
    f'Preparing {len(manifest)} proteins'
)

print(
    'raw_pdb_dir =',
    opts['raw_pdb_dir']
)

print(
    'esm_dir =',
    opts['esm_dir']
)


# ============================================================
# Per-protein preprocessing
# ============================================================

prepared = 0
cached = 0


for i, row in enumerate(
    manifest.itertuples(),
    start=1
):

    pid = str(row.prot_id)
    pair = f'{pid}:A:X1'

    fp = (
        root
        /
        'data_label'
        /
        pair
    )

    mp = (
        root
        /
        'data_label'
        /
        f'{pair}.surface_map.json'
    )


    # --------------------------------------------------------
    # Cache reuse
    # --------------------------------------------------------

    if (
        fp.exists()
        and
        mp.exists()
        and
        not a.rebuild
    ):
        cached += 1

        print(
            f'[{i}/{len(manifest)}] {pid}: cached'
        )

        continue


    print(
        f'[{i}/{len(manifest)}] {pid}: START'
    )


    # --------------------------------------------------------
    # Strict input checks
    # --------------------------------------------------------

    pdb_path = Path(row.pdb_path)
    esm_path = Path(row.esm_txt)

    if not pdb_path.exists():
        raise FileNotFoundError(
            f'{pid}: PDB missing: {pdb_path}'
        )

    if not esm_path.exists():
        raise FileNotFoundError(
            f'{pid}: ESM text missing: {esm_path}'
        )


    try:

        # ----------------------------------------------------
        # AUTHORS' processor
        # ----------------------------------------------------

        proc = MetalBindProcessor(
            pair=pair,
            anno='',
            ligand_type='ZN',
            # Metal branch -> 0.5-A probe radius in author's code
            dir_opts=opts,
            RQ_window=6.0,
            # exact public prepare_one.sh setting
        )

        proc.get_data()


        # ----------------------------------------------------
        # Verify author's data object
        # ----------------------------------------------------

        if not fp.exists():
            raise RuntimeError(
                f'{pid}: proc.get_data() finished but expected '
                f'data file was not created: {fp}'
            )


        # ----------------------------------------------------
        # Surface residue -> canonical sequence position
        # ----------------------------------------------------

        mapping = []

        for s in proc.surface_res:

            fields = s.split('_')

            if len(fields) < 2:
                raise RuntimeError(
                    f'{pid}: malformed surface residue: {s}'
                )

            if fields[0] != 'A':
                raise RuntimeError(
                    f'{pid}: unexpected surface chain {fields[0]}'
                )

            resid = int(fields[1])

            if (
                resid < 1
                or
                resid > len(row.sequence)
            ):
                raise RuntimeError(
                    f'{pid}: surface residue number {resid} '
                    f'outside sequence length {len(row.sequence)}'
                )

            mapping.append(
                {
                    'surface_residue': s,
                    'canonical_index': resid - 1,
                }
            )


        if len(mapping) == 0:
            raise RuntimeError(
                f'{pid}: no surface residues produced'
            )


        # ----------------------------------------------------
        # Save exact remapping metadata
        # ----------------------------------------------------

        mp.write_text(
            json.dumps(
                {
                    'prot_id': pid,
                    'sequence_length': len(row.sequence),
                    'surface_res': list(
                        proc.surface_res
                    ),
                    'mapping': mapping,
                },
                indent=2
            )
        )


        if not mp.exists():
            raise RuntimeError(
                f'{pid}: surface map was not written'
            )


        prepared += 1

        print(
            f'[{i}/{len(manifest)}] {pid}: PASS '
            f'surface_res={len(mapping)}'
        )


    except Exception:

        print(
            '\n'
            + '=' * 80
        )

        print(
            f'FAILED PROTEIN: {pid}'
        )

        print(
            f'PAIR: {pair}'
        )

        print(
            f'PDB: {pdb_path}'
        )

        print(
            f'ESM: {esm_path}'
        )

        print(
            '=' * 80
        )

        traceback.print_exc()

        print(
            '=' * 80
            + '\n'
        )

        raise


print(
    f'PREPROCESS COMPLETE: '
    f'prepared_now={prepared}, '
    f'cached={cached}, '
    f'total={len(manifest)}'
)
''')


# ============================================================
# RUN PREPROCESSING + STRICT COMPLETENESS AUDIT
# ============================================================

if RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE:

    for dataset, df in DATASETS.items():

        print(
            '\n'
            + '=' * 80
        )

        print(
            'MetalBind surface preprocessing:',
            dataset
        )

        print(
            '=' * 80
        )


        root = (
            METALB_PREP_ROOT
            /
            dataset
        )

        manifest_path = (
            root
            /
            'manifest.csv'
        )


        if not manifest_path.exists():

            raise FileNotFoundError(
                f'{dataset}: manifest missing: '
                f'{manifest_path}. Run 18C first.'
            )


        cmd = (
            f'python "{METALB_PREP_SCRIPT}" '
            f'--manifest "{manifest_path}" '
            f'--root "{root}"'
            +
            (
                ' --rebuild'
                if REBUILD_METALBIND_SURFACE
                else ''
            )
        )


        # IMPORTANT:
        # capture real subprocess traceback instead of hiding it
        # behind CalledProcessError.
        rr = metal_preprocess_mex(
            cmd,
            cwd=METALB,
            check=False,
            capture=True
        )


        print(
            '\n'
            + '-' * 80
        )

        print(
            f'OUTPUT: {dataset}'
        )

        print(
            '-' * 80
        )


        if rr.stdout:
            print(
                rr.stdout
            )


        print(
            f'\nRETURN CODE: {rr.returncode}'
        )


        if rr.returncode != 0:

            raise RuntimeError(
                f'{dataset}: MetalBind surface preprocessing '
                f'failed with return code {rr.returncode}. '
                f'The real subprocess traceback is printed above.'
            )


        # ====================================================
        # STRICT POST-PREPROCESSING COMPLETENESS AUDIT
        # ====================================================

        missing = []
        bad_maps = []


        for r in df.itertuples():

            pid = str(
                r.prot_id
            )

            pair = (
                f'{pid}:A:X1'
            )

            data_fp = (
                root
                /
                'data_label'
                /
                pair
            )

            map_fp = (
                root
                /
                'data_label'
                /
                f'{pair}.surface_map.json'
            )


            if not data_fp.exists():

                missing.append(
                    (
                        pid,
                        'pickle'
                    )
                )


            if not map_fp.exists():

                missing.append(
                    (
                        pid,
                        'map'
                    )
                )

                continue


            try:

                z = json.loads(
                    map_fp.read_text()
                )

            except Exception as e:

                bad_maps.append(
                    (
                        pid,
                        f'unreadable map: {e}'
                    )
                )

                continue


            if (
                int(
                    z.get(
                        'sequence_length',
                        -1
                    )
                )
                !=
                len(r.sequence)
            ):

                bad_maps.append(
                    (
                        pid,
                        'sequence_length mismatch'
                    )
                )


            mapping = z.get(
                'mapping',
                []
            )


            if len(mapping) == 0:

                bad_maps.append(
                    (
                        pid,
                        'empty mapping'
                    )
                )

                continue


            canonical_indices = [
                int(
                    x[
                        'canonical_index'
                    ]
                )
                for x
                in mapping
            ]


            if any(
                x < 0
                or
                x >= len(r.sequence)
                for x
                in canonical_indices
            ):

                bad_maps.append(
                    (
                        pid,
                        'canonical index out of range'
                    )
                )


        if missing:

            raise RuntimeError(
                f'{dataset}: MetalBind preprocessing incomplete: '
                f'{missing[:10]}'
            )


        if bad_maps:

            raise RuntimeError(
                f'{dataset}: MetalBind surface-map audit failed: '
                f'{bad_maps[:10]}'
            )


        print(
            f'{dataset}: MetalBind preprocessing PASS '
            f'{len(df)}/{len(df)} proteins.'
        )


    print(
        '\n'
        'MetalBind surface preprocessing '
        'PASSED for TestSet300 + TestSetNew46.'
    )
if RUN_METALBIND_TRANSFER and METALBIND_FINAL_CACHE_COMPLETE:
    print('18D preprocessing: SKIP (final MetalBind ion predictions already complete).')



MetalBind surface preprocessing: TestSet300
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/repos/metalbind/external_prepare_v6.py" --manifest "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/TestSet300/manifest.csv" --root "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/TestSet300"

--------------------------------------------------------------------------------
OUTPUT: TestSet300
--------------------------------------------------------------------------------
Preparing 300 proteins
raw_pdb_dir = /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/TestSet300/raw_pdb
esm_dir = /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/TestSet300/600M

## 11. All 13 released MetalBind checkpoints


In [22]:
# ============================================================
# 18E. RUN ALL 13 OFFICIAL MetalBind CHECKPOINTS AND REMAP FULL SEQUENCES
# ============================================================
METALB_INFER_SCRIPT=METALB/'external_infer_v6.py'
METALB_INFER_SCRIPT.write_text(r'''
import argparse, ast, json
from pathlib import Path
import _pickle as cPickle
import numpy as np
import pandas as pd
import torch

from Arguments import parser as author_parser
from models.MetalBind_model import MetalBind

ap=argparse.ArgumentParser()
ap.add_argument('--manifest',required=True)
ap.add_argument('--prepared_root',required=True)
ap.add_argument('--repo',required=True)
ap.add_argument('--output_root',required=True)
ap.add_argument('--rebuild',action='store_true')
a=ap.parse_args()

if not torch.cuda.is_available():
    raise RuntimeError('MetalBind author forward path requires CUDA in this reproduction')

manifest=pd.read_csv(a.manifest)
prep=Path(a.prepared_root)
repo=Path(a.repo)
outroot=Path(a.output_root); outroot.mkdir(parents=True,exist_ok=True)

ions=['CA','CD','CO','CU','CU1','FE','FE2','K','MG','MN','NA','NI','ZN']


def cached_ion_valid(path):
    if not path.exists() or a.rebuild:
        return False
    try:
        z=pd.read_csv(path); z['prot_id']=z['prot_id'].astype(str)
        need={'prot_id','residue_index','probability','is_surface_residue'}
        if not need.issubset(z.columns):
            return False
        if len(z)!=sum(len(str(x.sequence)) for x in manifest.itertuples()):
            return False
        for row in manifest.itertuples():
            pid=str(row.prot_id); L=len(row.sequence)
            q=z[z.prot_id.eq(pid)].sort_values('residue_index')
            if len(q)!=L or not np.array_equal(q.residue_index.to_numpy(int),np.arange(L)):
                return False
            p=q.probability.to_numpy(float)
            if not np.all(np.isfinite(p)) or np.min(p)<0 or np.max(p)>1:
                return False
        return True
    except Exception:
        return False


def tensor_data(data):
    n_points=data['xyz'].shape[0]
    xyz_type=np.asarray(data['xyz_type']).reshape(-1,1).astype(np.int64)
    chemi=torch.zeros((n_points,6),dtype=torch.float32)
    chemi.scatter_(1,torch.from_numpy(xyz_type),1.0)
    return {
        'xyz':torch.from_numpy(np.asarray(data['xyz'],dtype=np.float32)),
        'nuv':torch.from_numpy(np.asarray(data['nuv'],dtype=np.float32)).contiguous(),
        'hmm':torch.from_numpy(np.asarray(data['esm'],dtype=np.float32)),
        'geo':torch.from_numpy(np.asarray(data['pka'],dtype=np.float32)),
        'chemi':chemi,
        'y':torch.zeros(n_points,dtype=torch.long),
        'site':torch.zeros(len(data['site']),dtype=torch.long),
        'site_point':torch.from_numpy(np.asarray(data['site_point'],dtype=np.int64)),
        'xyz_batch':torch.zeros(n_points,dtype=torch.long),
    }

for ion in ions:
    cached_fp=outroot/f'{ion}.csv.gz'
    if cached_ion_valid(cached_fp):
        print(ion,'cached -> SKIP checkpoint inference')
        continue

    ckpt=repo/'Pretrained_model'/ion
    if not (ckpt/'best.pth').exists():
        raise FileNotFoundError(f'missing released checkpoint {ckpt/"best.pth"}')

    args=author_parser.parse_args([
        '--ligand',ion,
        '--checkpoints_dir',str(ckpt),
        '--device','cuda:0',
        '--radius','6.0',
        '--n_layers','4',
        '--input_feature_type','esm','chemi','pka',
        '--batch_size','1',
        '--lr','0.001',
        '--loss_type','site',
    ])
    # dir_opts is not consumed by model construction/inference once tensors are ready.
    args.dir_opts={}
    model=MetalBind(args)
    model.load_network()
    model.eval()

    rows=[]
    with torch.no_grad():
        for row in manifest.itertuples():
            pid=str(row.prot_id); L=len(row.sequence)
            pair=f'{pid}:A:X1'
            with open(prep/'data_label'/pair,'rb') as f:
                data=cPickle.load(f)
            meta=json.loads((prep/'data_label'/f'{pair}.surface_map.json').read_text())

            td=tensor_data(data)
            model.set_input(td)
            surf=np.asarray(model.test().detach().cpu(),dtype=float).reshape(-1)
            mapping=meta['mapping']
            if len(surf) != len(mapping):
                raise RuntimeError(f'{pid}/{ion}: model emitted {len(surf)} surface residues, map has {len(mapping)}')
            if not np.all(np.isfinite(surf)) or np.min(surf)<0 or np.max(surf)>1:
                raise RuntimeError(f'{pid}/{ion}: invalid surface probabilities')

            full=np.zeros(L,dtype=np.float32)
            seen=set()
            for prob,m in zip(surf,mapping):
                idx=int(m['canonical_index'])
                if idx in seen:
                    raise RuntimeError(f'{pid}: duplicate canonical surface index {idx}')
                seen.add(idx)
                full[idx]=float(prob)

            for i,p in enumerate(full):
                rows.append((pid,i,float(p),i in seen))

    out=pd.DataFrame(rows,columns=['prot_id','residue_index','probability','is_surface_residue'])
    out.to_csv(outroot/f'{ion}.csv.gz',index=False,compression='gzip')
    print(ion,'saved',len(out),'residue rows')
    del model
    torch.cuda.empty_cache()
''')

if RUN_METALBIND_TRANSFER and not METALBIND_FINAL_CACHE_COMPLETE:
    for dataset,df in DATASETS.items():
        root=METALB_PREP_ROOT/dataset
        outroot=METALBIND_EXTERNAL_ROOT/dataset
        outroot.mkdir(parents=True,exist_ok=True)
        cmd=(
            f'python "{METALB_INFER_SCRIPT}" --manifest "{root/"manifest.csv"}" '
            f'--prepared_root "{root}" --repo "{METALB}" --output_root "{outroot}"'
            + (' --rebuild' if REBUILD_METALBIND_INFERENCE else '')
        )
        rr=metal_mex(cmd,cwd=METALB,check=False,capture=True)
        if rr.stdout:
            print(rr.stdout)
        if rr.returncode!=0:
            raise RuntimeError(
                f'{dataset}: MetalBind inference failed with return code {rr.returncode}. '
                'The real subprocess traceback is printed above.'
            )
elif RUN_METALBIND_TRANSFER:
    print('18E MetalBind inference: SKIP (all 13 ion outputs already strictly cached).')


18E MetalBind inference: SKIP (all 13 ion outputs already strictly cached).


## 12. Pool 13 ions + exact common metrics


In [23]:
# ============================================================
# 18F. STRICT FULL-LENGTH MetalBind INGEST + POOL + FULL METRICS
# ============================================================
def load_metalbind_ion_csv(path,df):
    z=pd.read_csv(path)
    need={'prot_id','residue_index','probability','is_surface_residue'}
    if not need.issubset(z.columns):
        raise ValueError(f'{path}: need columns {sorted(need)}')
    z['prot_id']=z['prot_id'].astype(str)
    out={}
    for r in df.itertuples():
        q=z[z.prot_id.eq(r.prot_id)].sort_values('residue_index')
        if len(q)!=len(r.sequence):
            raise RuntimeError(f'{path}: {r.prot_id}: {len(q)} rows != {len(r.sequence)} residues')
        idx=q.residue_index.to_numpy(int)
        if not np.array_equal(idx,np.arange(len(r.sequence))):
            raise RuntimeError(f'{path}: noncanonical indexing for {r.prot_id}')
        p=q.probability.to_numpy(float)
        if not np.all(np.isfinite(p)) or np.min(p)<0 or np.max(p)>1:
            raise RuntimeError(f'{path}: invalid probabilities for {r.prot_id}')
        out[r.prot_id]=p
    return out


metalb_rows=[]
metalbind_pooled_probs={}
metalbind_ion_probs={}

if RUN_METALBIND_TRANSFER:
    surface_coverage=[]
    for dataset,df in DATASETS.items():
        ion_probs={}
        for ion in METALB_IONS:
            fp=METALBIND_EXTERNAL_ROOT/dataset/f'{ion}.csv.gz'
            if not fp.exists():
                raise FileNotFoundError(f'MetalBind missing completed ion file: {fp}')
            ion_probs[ion]=load_metalbind_ion_csv(fp,df)

        pooled={}
        for r in df.itertuples():
            pooled[r.prot_id]=np.maximum.reduce([ion_probs[ion][r.prot_id] for ion in METALB_IONS])
        metalbind_pooled_probs[dataset]=pooled
        metalbind_ion_probs[dataset]=ion_probs

        save_probs(pooled,PRED/f'MetalBind_13ion_max_TRANSFER__{dataset}__metal_binding.csv.gz')
        out,pt=evaluate_probability_dict(
            df,pooled,'metal_binding','MetalBind_13ion_max_TRANSFER',dataset,
            'fixed_0.5',0.5,'C_cross_benchmark_transfer'
        )
        metalb_rows.append(out)
        pt.to_csv(RESULTS/f'MetalBind_{dataset}_per_protein.csv',index=False)

        # Surface coverage is method-internal context, not an evaluation filter.
        z=pd.read_csv(METALBIND_EXTERNAL_ROOT/dataset/f'{METALB_IONS[0]}.csv.gz')
        for pid,q in z.groupby(z.prot_id.astype(str)):
            surface_coverage.append([dataset,pid,int(q.is_surface_residue.astype(bool).sum()),len(q),float(q.is_surface_residue.astype(bool).mean())])

    metalb_df=pd.DataFrame(metalb_rows)
    metalb_df.to_csv(RESULTS/'MetalBind_TRANSFER_FULL_METRICS_300_46.csv',index=False)
    pd.DataFrame(surface_coverage,columns=['dataset','prot_id','surface_residues','total_residues','surface_fraction']).to_csv(
        RESULTS/'MetalBind_surface_coverage_300_46.csv',index=False
    )
    display(metalb_df)


,method,dataset,label,comparison_tier,operating_point,n_residues,n_positive,n_negative,prevalence,AP_AUPRC,...,protein_F1_bootstrap95_low,protein_F1_bootstrap95_high,protein_mean_MCC,protein_MCC_CI95_legacy_halfwidth,protein_MCC_CI95_standard_halfwidth,protein_MCC_bootstrap95_low,protein_MCC_bootstrap95_high,protein_mean_accuracy,protein_accuracy_CI95_legacy_halfwidth,protein_accuracy_CI95_standard_halfwidth
0,MetalBind_13ion_max_TRANSFER,TestSet300,metal_binding,C_cross_benchmark_transfer,fixed_0.5,62689,881,61808,0.014054,0.202519,...,0.158520,0.224811,0.192436,0.033476,0.033532,0.159940,0.228125,0.951567,0.004284,0.004291
1,MetalBind_13ion_max_TRANSFER,TestSetNew46,metal_binding,C_cross_benchmark_transfer,fixed_0.5,6227,77,6150,0.012366,0.342315,...,0.104104,0.271307,0.192525,0.091201,0.092264,0.109168,0.281326,0.928249,0.014389,0.014556


In [25]:
# ============================================================
# 13. FINAL METALBIND-ONLY RESULT READER + HARD COMPLETION GUARD
# ============================================================
metrics_fp = RESULTS/'MetalBind_TRANSFER_FULL_METRICS_300_46.csv'

if not metrics_fp.exists():
    raise FileNotFoundError(
        f'Expected final MetalBind metric file was not created: {metrics_fp}'
    )

metalbind_final = pd.read_csv(metrics_fp)

expected_datasets = {'TestSet300', 'TestSetNew46'}
got_datasets = set(metalbind_final['dataset'].astype(str))

if not expected_datasets.issubset(got_datasets):
    raise RuntimeError(
        f'MetalBind final metrics missing datasets: '
        f'{sorted(expected_datasets - got_datasets)}'
    )

bad_ion_files = []

for dataset, df in DATASETS.items():
    for ion in METALB_IONS:
        fp = METALBIND_EXTERNAL_ROOT/dataset/f'{ion}.csv.gz'

        if not _simple_prob_csv_complete(
            fp, df, require_surface_column=True
        ):
            bad_ion_files.append(str(fp))

if bad_ion_files:
    raise RuntimeError(
        'MetalBind completion guard failed; invalid/missing ion outputs:\n'
        + '\n'.join(bad_ion_files[:20])
    )

preferred_cols = [
    'dataset',
    'n_residues',
    'n_positive',
    'prevalence',
    'AP_AUPRC',
    'PR_AUC_trapezoid',
    'AUROC',
    'global_precision_PPV',
    'global_recall_sensitivity',
    'global_specificity',
    'global_positive_F1',
    'global_MCC',
    'protein_mean_precision',
    'protein_mean_recall',
    'protein_mean_specificity',
    'protein_mean_F1',
    'protein_mean_MCC',
]

show_cols = [
    c for c in preferred_cols
    if c in metalbind_final.columns
]

compact = metalbind_final[show_cols].copy()

print('\n' + '='*88)
print('METALBIND FINAL RESULTS')
print('='*88)
display(compact)

coverage_fp = RESULTS/'MetalBind_surface_coverage_300_46.csv'

if coverage_fp.exists():
    coverage = pd.read_csv(coverage_fp)

    coverage_summary = (
        coverage
        .groupby('dataset')
        .agg(
            proteins=('prot_id','nunique'),
            mean_surface_fraction=('surface_fraction','mean'),
            median_surface_fraction=('surface_fraction','median'),
        )
        .reset_index()
    )

    print('\nSurface coverage context:')
    display(coverage_summary)

print('\nSaved full metrics:')
print(metrics_fp)

print('\nSaved pooled residue probabilities:')
for dataset in DATASETS:
    print(
        PRED/f'MetalBind_13ion_max_TRANSFER__{dataset}__metal_binding.csv.gz'
    )

print('\n' + '='*88)
print(
    'METALBIND TRANSFER REPRODUCTION COMPLETE: '
    '2/2 datasets, 13/13 released ion models each.'
)
print('='*88)

print(
    'Interpretation: this completes the locked-checkpoint MetalBind transfer '
    'evaluation on TestSet300 and TestSetNew46 used by the master comparison. '
    "It is not a claim of reproducing MetalBind's original native benchmark."
)



METALBIND FINAL RESULTS


,dataset,n_residues,n_positive,prevalence,AP_AUPRC,PR_AUC_trapezoid,AUROC,global_precision_PPV,global_recall_sensitivity,global_specificity,global_positive_F1,global_MCC,protein_mean_precision,protein_mean_recall,protein_mean_specificity,protein_mean_F1,protein_mean_MCC
0,TestSet300,62689,881,0.014054,0.202519,0.201822,0.826286,0.179752,0.493757,0.967884,0.263556,0.282072,0.170232,0.255771,0.958431,0.189624,0.192436
1,TestSetNew46,6227,77,0.012366,0.342315,0.336930,0.926658,0.151515,0.779221,0.945366,0.253700,0.328136,0.152505,0.274068,0.929307,0.184464,0.192525



Surface coverage context:


,dataset,proteins,mean_surface_fraction,median_surface_fraction
0,TestSet300,300,0.999736,1.0
1,TestSetNew46,46,0.999677,1.0



Saved full metrics:
/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/results/MetalBind_TRANSFER_FULL_METRICS_300_46.csv

Saved pooled residue probabilities:
/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/predictions/MetalBind_13ion_max_TRANSFER__TestSet300__metal_binding.csv.gz
/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/predictions/MetalBind_13ion_max_TRANSFER__TestSetNew46__metal_binding.csv.gz

METALBIND TRANSFER REPRODUCTION COMPLETE: 2/2 datasets, 13/13 released ion models each.
Interpretation: this completes the locked-checkpoint MetalBind transfer evaluation on TestSet300 and TestSetNew46 used by the master comparison. It is not a claim of reproducing MetalBind's original native benchmark.
